In [2]:
"""
Simulate a 7-DOF Franka Emika Panda in MuJoCo.

Setup:
    pip install mujoco numpy
    git clone https://github.com/google-deepmind/mujoco_menagerie.git

Run (opens the interactive viewer):
    python franka_sim.py --demo joint     # each joint follows a sine wave
    python franka_sim.py --demo ik        # end effector traces a circle (IK)

Run without a window (for servers / testing):
    python franka_sim.py --demo ik --headless

macOS note: the viewer needs `mjpython franka_sim.py ...` instead of `python`.
"""
import argparse
import time
from pathlib import Path

import mujoco
import mujoco.viewer
import numpy as np

DEFAULT_MODEL = "mujoco_menagerie/franka_emika_panda/scene.xml"
TCP_OFFSET = np.array([0.0, 0.0, 0.1034])  # tool-center point, in the "hand" frame
ARM_JOINTS = [f"joint{i}" for i in range(1, 8)]


class FrankaSim:
    def __init__(self, model_path: str):
        self.model = mujoco.MjModel.from_xml_path(str(model_path))
        self.data = mujoco.MjData(self.model)

        # Joint / DOF / actuator indices for the 7 arm joints
        jids = [self.model.joint(n).id for n in ARM_JOINTS]
        self.qpos_idx = [self.model.jnt_qposadr[j] for j in jids]
        self.dof_idx = [self.model.jnt_dofadr[j] for j in jids]
        self.hand_id = self.model.body("hand").id

        # Actuator ctrl limits (position targets for joints 1-7)
        self.ctrl_lo = self.model.actuator_ctrlrange[:7, 0].copy()
        self.ctrl_hi = self.model.actuator_ctrlrange[:7, 1].copy()

        self.reset()

    # ---------- state helpers ----------
    def reset(self):
        """Reset to the 'home' keyframe defined in panda.xml."""
        key = self.model.key("home").id
        mujoco.mj_resetDataKeyframe(self.model, self.data, key)
        mujoco.mj_forward(self.model, self.data)
        self.q_target = self.data.qpos[self.qpos_idx].copy()

    @property
    def q(self):
        return self.data.qpos[self.qpos_idx].copy()

    def ee_pos(self):
        """World position of the tool-center point."""
        R = self.data.xmat[self.hand_id].reshape(3, 3)
        return self.data.xpos[self.hand_id] + R @ TCP_OFFSET

    # ---------- control ----------
    def set_joint_targets(self, q_des):
        """Actuators are position servos: ctrl = desired joint angle (rad)."""
        self.data.ctrl[:7] = np.clip(q_des, self.ctrl_lo, self.ctrl_hi)

    def set_gripper(self, opening: float):
        """opening in [0, 1]; 0 = closed, 1 = fully open (actuator8 range is 0-255)."""
        self.data.ctrl[7] = 255.0 * float(np.clip(opening, 0, 1))

    def ik_step(self, target_pos, gain=0.05, damping=1e-2, max_dq=0.003):
        """One damped-least-squares differential IK step (position only).
        Called every physics step (500 Hz), so gain and max_dq are small per-step values.
        Updates self.q_target toward target_pos and returns it."""
        jacp = np.zeros((3, self.model.nv))
        point = self.ee_pos()
        mujoco.mj_jac(self.model, self.data, jacp, None, point, self.hand_id)
        J = jacp[:, self.dof_idx]  # 3 x 7

        err = target_pos - point
        # dq = J^T (J J^T + lambda I)^-1 * err
        dq = J.T @ np.linalg.solve(J @ J.T + damping * np.eye(3), gain * err)
        dq = np.clip(dq, -max_dq, max_dq)

        self.q_target = np.clip(self.q_target + dq, self.ctrl_lo, self.ctrl_hi)
        return self.q_target

    def step(self):
        mujoco.mj_step(self.model, self.data)


# ---------- demos ----------
def demo_joint(sim: FrankaSim, t: float):
    """Small sine wave around the home pose on every joint."""
    home = sim.model.key("home").qpos[sim.qpos_idx]
    amp = np.array([0.4, 0.3, 0.4, 0.3, 0.4, 0.3, 0.5])
    freq = 0.3  # Hz
    sim.set_joint_targets(home + amp * np.sin(2 * np.pi * freq * t))
    sim.set_gripper(0.5 * (1 + np.sin(2 * np.pi * 0.2 * t)))


def make_demo_ik(sim: FrankaSim):
    center = sim.ee_pos().copy()
    radius = 0.10

    def demo_ik(sim: FrankaSim, t: float):
        # circle in the plane x = const (y-z plane), 0.1 m radius
        target = center + radius * np.array([0.0, np.sin(0.8 * t), np.cos(0.8 * t) - 1.0])
        sim.set_joint_targets(sim.ik_step(target))

    return demo_ik


def run(args):
    model_path = Path(args.model)
    if not model_path.exists():
        raise SystemExit(
            f"Model not found: {model_path}\n"
            "Clone it with:\n"
            "  git clone https://github.com/google-deepmind/mujoco_menagerie.git"
        )

    sim = FrankaSim(model_path)
    print(f"nq={sim.model.nq}  nv={sim.model.nv}  nu={sim.model.nu}  dt={sim.model.opt.timestep}")
    print("Start EE position:", np.round(sim.ee_pos(), 4))

    demo = demo_joint if args.demo == "joint" else make_demo_ik(sim)

    if args.headless:
        while sim.data.time < args.duration:
            demo(sim, sim.data.time)
            sim.step()
        print("Final joint angles:", np.round(sim.q, 3))
        print("Final EE position :", np.round(sim.ee_pos(), 4))
        return

    with mujoco.viewer.launch_passive(sim.model, sim.data) as viewer:
        wall_start = time.time()
        while viewer.is_running() and sim.data.time < args.duration:
            demo(sim, sim.data.time)
            sim.step()
            viewer.sync()
            # keep sim in real time
            lag = sim.data.time - (time.time() - wall_start)
            if lag > 0:
                time.sleep(lag)


if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--model", default=DEFAULT_MODEL)
    p.add_argument("--demo", choices=["joint", "ik"], default="joint")
    p.add_argument("--duration", type=float, default=60.0, help="sim seconds")
    p.add_argument("--headless", action="store_true")
    args, _ = p.parse_known_args()   # ignores the extra arguments Jupyter adds
    run(args)

nq=9  nv=9  nu=8  dt=0.002
Start EE position: [ 0.5545 -0.      0.5211]
